In [3]:
# !pip install plotly

In [5]:
import os
import json
import pandas as pd
from google import genai
from pydantic import BaseModel, Field
import plotly.express as px
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# 1. Define the target structure for Gemini
class Transaction(BaseModel):
    date: str = Field(description="Transaction date in YYYY-MM-DD format")
    vendor: str = Field(description="Name of the vendor or client")
    amount: float = Field(description="Total amount of the transaction")
    type: str = Field(description="Must be exactly 'Income' or 'Expense'")
    category: str = Field(description="Categorize this as: Software, Travel, Payroll, Meals, Office Supplies, or Marketing")

class FinancialDocument(BaseModel):
    transactions: list[Transaction]

# 2. Load your saved JSON file
# Replace 'extracted_ocr_data.json' with your actual file path
with open('ai_cfo_database.json', 'r') as file:
    raw_json_data = json.load(file)

# Convert the loaded JSON back into a string so Gemini can read it as context
prompt_context = json.dumps(raw_json_data)

# 3. Initialize Gemini and analyze the text
client = genai.Client() # Assumes GEMINI_API_KEY is in your environment variables

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=f"Analyze this raw OCR JSON data. Extract, clean, and categorize all financial transactions: {prompt_context}",
    config={
        'response_mime_type': 'application/json',
        'response_schema': FinancialDocument,
        'temperature': 0.1 
    },
)

# 4. Convert Gemini's structured output into a Pandas DataFrame
structured_data = response.parsed
records = [t.model_dump() for t in structured_data.transactions]
df = pd.DataFrame(records)

# Ensure data types are correct
df['date'] = pd.to_datetime(df['date'])
df['amount'] = pd.to_numeric(df['amount'])

print("Data successfully structured!")
print(df.head())

# 5. Visualize the Data
def plot_expense_breakdown(df):
    expenses = df[df['type'] == 'Expense']
    fig = px.pie(expenses, values='amount', names='category', title='Expense Breakdown')
    fig.show()

plot_expense_breakdown(df)

Data successfully structured!
        date                                     vendor    amount     type  \
0 2023-08-02  Konde Products & Services Private Limited   1899.00  Expense   
1 2023-08-02  Konde Products & Services Private Limited   1899.00  Expense   
2 2026-02-17                          RIO KITCHEN & BAR  21195.25  Expense   

    category  
0  Marketing  
1  Marketing  
2      Meals  


In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# 1. Load your newly structured JSON file
with open('ai_cfo_database.json', 'r') as file:
    raw_json_data = json.load(file)

# 2. Flatten the nested JSON into a Pandas DataFrame
df = pd.json_normalize(raw_json_data)

# Ensure numeric columns are treated as numbers
df['financial_totals.grand_total'] = pd.to_numeric(df['financial_totals.grand_total'])
df['financial_totals.total_tax_amount'] = pd.to_numeric(df['financial_totals.total_tax_amount'])
df['financial_totals.taxable_amount_subtotal'] = pd.to_numeric(df['financial_totals.taxable_amount_subtotal'])

print("✅ Data successfully flattened for multiple plots:")
display(df[['document_metadata.invoice_date', 'seller_details.platform_or_marketplace', 'financial_totals.grand_total']].head())

# --- PLOT 1: Spend by Category (Pie Chart) ---
def plot_category_breakdown(df):
    # Group by category in case there are multiple of the same category
    category_df = df.groupby('document_metadata.expense_category')['financial_totals.grand_total'].sum().reset_index()
    
    fig = px.pie(
        category_df, 
        values='financial_totals.grand_total', 
        names='document_metadata.expense_category', 
        title='1. Expense Breakdown by Category',
        color_discrete_sequence=px.colors.sequential.Plasma
    )
    fig.show()

# --- PLOT 2: Spend by Vendor (Bar Chart) ---
def plot_vendor_spend(df):
    vendor_df = df.groupby('seller_details.platform_or_marketplace')['financial_totals.grand_total'].sum().reset_index()
    
    fig = px.bar(
        vendor_df, 
        x='seller_details.platform_or_marketplace', 
        y='financial_totals.grand_total',
        title='2. Total Spend by Vendor / Platform',
        labels={'seller_details.platform_or_marketplace': 'Vendor', 'financial_totals.grand_total': 'Total Spent (INR)'},
        text_auto=True
    )
    fig.show()

# --- PLOT 3: Tax vs Base Amount (Stacked Bar) ---
def plot_tax_breakdown(df):
    # Sum up the total taxable base and the total tax
    total_taxable = df['financial_totals.taxable_amount_subtotal'].sum()
    total_tax = df['financial_totals.total_tax_amount'].sum()
    
    # Create a simple figure using graph_objects for custom control
    fig = go.Figure(data=[
        go.Bar(name='Taxable Base Amount', x=['Overall Spend'], y=[total_taxable]),
        go.Bar(name='Total Tax Paid', x=['Overall Spend'], y=[total_tax])
    ])
    
    fig.update_layout(
        title='3. Tax vs. Base Amount Breakdown',
        barmode='stack',
        yaxis_title="Amount (INR)"
    )
    fig.show()

# 4. Generate all three plots
plot_category_breakdown(df)
plot_vendor_spend(df)
plot_tax_breakdown(df)

✅ Data successfully flattened for multiple plots:


,document_metadata.invoice_date,seller_details.platform_or_marketplace,financial_totals.grand_total
0,2023-08-02,Myntra,1899.00
1,2023-08-02,Myntra,1899.00
2,2023-06-25,None,5775.36
3,2026-02-17,NaN,21195.25
4,2025-03-29,None,3565.53


In [17]:
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# 1. Load and flatten the JSON data
with open('extracted_ocr_data.json', 'r') as file:
    raw_json_data = json.load(file)

df = pd.json_normalize(raw_json_data)

# Ensure our financial columns are treated as numbers (forces any errors to NaN, then fills with 0)
for col in ['financial_totals.grand_total', 'financial_totals.total_tax_amount', 'financial_totals.taxable_amount_subtotal']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 2. Aggregate the Data for the plots
cat_df = df.groupby('document_metadata.expense_category')['financial_totals.grand_total'].sum().reset_index()
ven_df = df.groupby('seller_details.platform_or_marketplace')['financial_totals.grand_total'].sum().reset_index()
tax_df = df.groupby('seller_details.platform_or_marketplace')[['financial_totals.taxable_amount_subtotal', 'financial_totals.total_tax_amount']].sum().reset_index()

# 3. Create the Dashboard Layout (Grid)
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}],       # Top row: Pie chart, Bar chart
           [{"type": "xy", "colspan": 2}, None]],      # Bottom row: Wide stacked bar chart
    subplot_titles=("1. Expense Breakdown", "2. Top Vendors", "3. Tax vs. Base Spend (By Vendor)"),
    vertical_spacing=0.15,
    horizontal_spacing=0.10
)

# --- Top Left: Donut Chart (Category) ---
fig.add_trace(go.Pie(
    labels=cat_df['document_metadata.expense_category'],
    values=cat_df['financial_totals.grand_total'],
    hole=0.4, # Makes it a donut chart, which looks cleaner
    name="Category",
    hovertemplate="%{label}: ₹%{value:,.2f}<br>(%{percent})<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    labels=cat_df['document_metadata.expense_category'],
    values=cat_df['financial_totals.grand_total'],
    hole=0.4, # Makes it a donut chart, which looks cleaner
    name="Category",
    hovertemplate="%{label}: ₹%{value:,.2f}<br>(%{percent})<extra></extra>"
), row=1, col=1)


# --- Top Right: Bar Chart (Vendors) ---
fig.add_trace(go.Bar(
    x=ven_df['seller_details.platform_or_marketplace'],
    y=ven_df['financial_totals.grand_total'],
    name="Total Spend",
    marker_color='#636EFA',
    hovertemplate="Vendor: %{x}<br>Spend: ₹%{y:,.2f}<extra></extra>"
), row=1, col=2)

# --- Bottom Row: Stacked Bar (Tax vs Base by Vendor) ---
fig.add_trace(go.Bar(
    x=tax_df['seller_details.platform_or_marketplace'],
    y=tax_df['financial_totals.taxable_amount_subtotal'],
    name="Base Amount",
    marker_color='#00CC96',
    hovertemplate="Base: ₹%{y:,.2f}<extra></extra>"
), row=2, col=1)

fig.add_trace(go.Bar(
    x=tax_df['seller_details.platform_or_marketplace'],
    y=tax_df['financial_totals.total_tax_amount'],
    name="Tax Paid",
    marker_color='#EF553B',
    hovertemplate="Tax: ₹%{y:,.2f}<extra></extra>"
), row=2, col=1)

# 4. Polish the UI (Titles, Background, Axis Labels)
fig.update_layout(
    title_text="AI CFO Overview Dashboard",
    title_font_size=24,
    barmode='stack', # Stacks the bottom chart
    height=800,      # Give the dashboard enough height
    template='plotly_white', # Clean white background
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Format the Y-axes to show currency
fig.update_yaxes(title_text="Amount (INR)", tickprefix="₹", row=1, col=2)
fig.update_yaxes(title_text="Amount (INR)", tickprefix="₹", row=2, col=1)

fig.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.Scatter: 'labels'

Did you mean "legend"?

    Valid properties:
        alignmentgroup
            Set several traces linked to the same position axis or
            matching axes to the same alignmentgroup. This controls
            whether bars compute their positional range dependently
            or independently.
        cliponaxis
            Determines whether or not markers and text nodes are
            clipped about the subplot axes. To show markers and
            text nodes above axis lines and tick labels, make sure
            to set `xaxis.layer` and `yaxis.layer` to *below
            traces*.
        connectgaps
            Determines whether or not gaps (i.e. {nan} or missing
            values) in the provided data arrays are connected.
        customdata
            Assigns extra data each datum. This may be useful when
            listening to hover, click and selection events. Note
      

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# 1. Load and flatten the JSON data
with open('ai_cfo_database.json', 'r') as file:
    raw_json_data = json.load(file)

df = pd.json_normalize(raw_json_data)

# 2. Clean the Data
# Ensure financial columns are numbers
numeric_cols = [
    'financial_totals.grand_total', 
    'financial_totals.total_tax_amount', 
    'financial_totals.taxable_amount_subtotal'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Ensure dates are treated as actual time values for the timeline
df['document_metadata.invoice_date'] = pd.to_datetime(df['document_metadata.invoice_date'], errors='coerce')

print("✅ Data ready for 5-Plot AI CFO Dashboard!")

# --- PLOT 1: Expense Breakdown (Donut Chart) ---
def plot_category_donut(df):
    cat_df = df.groupby('document_metadata.expense_category')['financial_totals.grand_total'].sum().reset_index()
    fig = px.pie(
        cat_df, values='financial_totals.grand_total', names='document_metadata.expense_category',
        title='1. Expense Breakdown by Category', hole=0.4,
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    fig.update_traces(hovertemplate="%{label}: ₹%{value:,.2f}<br>(%{percent})")
    fig.show()

# --- PLOT 2: Top Vendors (Bar Chart) ---
def plot_vendor_bar(df):
    ven_df = df.groupby('seller_details.platform_or_marketplace')['financial_totals.grand_total'].sum().reset_index()
    ven_df = ven_df.sort_values(by='financial_totals.grand_total', ascending=False) # Sort highest to lowest
    fig = px.bar(
        ven_df, x='seller_details.platform_or_marketplace', y='financial_totals.grand_total',
        title='2. Total Spend by Vendor', labels={'seller_details.platform_or_marketplace': 'Vendor', 'financial_totals.grand_total': 'Total Spent (₹)'},
        text_auto='.2s'
    )
    fig.update_traces(marker_color='#636EFA')
    fig.show()

# --- PLOT 3: Cash Flow Timeline (Line Chart) ---
def plot_timeline(df):
    # Group by date and sort chronologically
    time_df = df.dropna(subset=['document_metadata.invoice_date']) # Remove missing dates
    time_df = time_df.groupby('document_metadata.invoice_date')['financial_totals.grand_total'].sum().reset_index()
    time_df = time_df.sort_values('document_metadata.invoice_date')
    
    fig = px.line(
        time_df, x='document_metadata.invoice_date', y='financial_totals.grand_total',
        title='3. Spending Over Time (Cash Flow Timeline)', markers=True,
        labels={'document_metadata.invoice_date': 'Date', 'financial_totals.grand_total': 'Amount Spent (₹)'}
    )
    fig.update_traces(line_color='#00CC96', line_width=3, marker=dict(size=8))
    fig.show()

# --- PLOT 4: Tax vs. Base Amount (Stacked Bar) ---
def plot_tax_stack(df):
    tax_df = df.groupby('seller_details.platform_or_marketplace')[['financial_totals.taxable_amount_subtotal', 'financial_totals.total_tax_amount']].sum().reset_index()
    
    fig = go.Figure(data=[
        go.Bar(name='Taxable Base Amount', x=tax_df['seller_details.platform_or_marketplace'], y=tax_df['financial_totals.taxable_amount_subtotal'], marker_color='#AB63FA'),
        go.Bar(name='Total Tax Paid', x=tax_df['seller_details.platform_or_marketplace'], y=tax_df['financial_totals.total_tax_amount'], marker_color='#EF553B')
    ])
    fig.update_layout(title='4. Tax vs. Base Amount by Vendor', barmode='stack', yaxis_title="Amount (₹)")
    fig.show()

# --- PLOT 5: Spend Hierarchy (Treemap) ---
def plot_spend_treemap(df):
    # Fills any missing vendors/categories with "Unknown" so the plot doesn't break
    tree_df = df.fillna({'document_metadata.expense_category': 'Unknown Category', 'seller_details.platform_or_marketplace': 'Unknown Vendor'})
    
    fig = px.treemap(
        tree_df, 
        path=[px.Constant("All Spend"), 'document_metadata.expense_category', 'seller_details.platform_or_marketplace'], 
        values='financial_totals.grand_total',
        title='5. Spend Hierarchy (Click a category to zoom in!)'
    )
    fig.update_traces(root_color="lightgrey")
    fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
    fig.show()

# 3. Generate all 5 plots
plot_category_donut(df)
plot_vendor_bar(df)
plot_timeline(df)
plot_tax_stack(df)
plot_spend_treemap(df)

✅ Data ready for 5-Plot AI CFO Dashboard!


In [19]:
# --- PLOT 6: Expense Comparison by Category (Bar Chart) ---
def plot_expense_category_bar(df):
    # Group by category and sum the totals
    cat_df = df.groupby('document_metadata.expense_category')['financial_totals.grand_total'].sum().reset_index()
    
    # Sort from highest expense to lowest
    cat_df = cat_df.sort_values(by='financial_totals.grand_total', ascending=False)
    
    fig = px.bar(
        cat_df, 
        x='document_metadata.expense_category', 
        y='financial_totals.grand_total',
        title='6. Expense Comparison by Category', 
        labels={
            'document_metadata.expense_category': 'Expense Category', 
            'financial_totals.grand_total': 'Total Spent (₹)'
        },
        text_auto='.2s', # Automatically adds the number on top of the bar
        color='document_metadata.expense_category' # Gives each category a distinct color
    )
    
    # Clean up the layout
    fig.update_layout(showlegend=False) # Hide legend since the x-axis already has the labels
    fig.show()

In [20]:
# 3. Generate all plots
plot_category_donut(df)
plot_vendor_bar(df)
plot_timeline(df)
plot_tax_stack(df)
plot_spend_treemap(df)

# Add your new plot here:
plot_expense_category_bar(df)

In [1]:
import google.generativeai as genai

model = genai.GenerativeModel(model_name="gemini-1.5-flash", api_version="v1beta")
models = genai.list_models()
for model in models:
   print(model.name)

C:\Users\Admin\AppData\Local\Temp\ipykernel_16572\3843298402.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


TypeError: GenerativeModel.__init__() got an unexpected keyword argument 'api_version'